# Misinformation Detection Pipeline
Inspired by FActScore. Pipeline:
1. Extract atomic facts from input text, encode with BERT, build a knowledge graph
2. Augment with Google Search evidence, ranked by cross-source agreement
3. Update knowledge graph with weighted evidence
4. Classify with a Graph Neural Network (GCN)

## Key design choices / fixes from v1:
- SentenceTransformer loaded once globally (not per call)
- Proper cosine similarity using sklearn (not torch, which needs explicit tensors+dim)
- Sentence splitting via nltk (handles abbreviations, decimals, etc.)
- Knowledge graph update no longer mutates G.nodes while iterating it
- Model saving via torch.save (not joblib, which is for sklearn)
- TF-IDF topic extraction implemented
- GCN classifier that actually uses the graph structure

In [31]:
# --- Global setup ---
# Load the sentence encoder ONCE. Reloading it inside every function call is very slow
# because it reads weights from disk each time.

import nltk
nltk.download('punkt_tab')

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity  # Correct: works on numpy arrays
import numpy as np

ENCODER = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim embeddings
EMBEDDING_DIM = 384

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bhada\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


## Step 1: Fact Extraction & Knowledge Graph Construction

In [32]:
import networkx as nx
from nltk.tokenize import sent_tokenize

def extract_facts(text: str) -> list[tuple[str, np.ndarray]]:
    """
    Split text into sentences and embed each one.
    Uses nltk sent_tokenize instead of split('.') to handle
    abbreviations like 'Dr. Smith' and decimal numbers correctly.
    Returns list of (sentence, embedding) tuples.
    """
    sentences = sent_tokenize(text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]  # filter fragments
    if not sentences:
        return []
    # Batch encoding is much faster than encoding one by one
    embeddings = ENCODER.encode(sentences, batch_size=32, show_progress_bar=False)
    return list(zip(sentences, embeddings))


def build_knowledge_graph(facts: list[tuple[str, np.ndarray]], threshold: float = 0.75) -> nx.Graph:
    """
    Build a graph where nodes are sentences and edges connect
    semantically similar sentences (cosine similarity > threshold).

    Note: edges here represent topical overlap, NOT logical support.
    A future improvement is to train a relation-extraction model to
    distinguish 'supports', 'contradicts', and 'neutral' edge types.
    """
    G = nx.Graph()
    if not facts:
        return G

    sentences, embeddings = zip(*facts)
    embeddings = np.array(embeddings)  # shape: (n, 384)

    for i, (sent, emb) in enumerate(zip(sentences, embeddings)):
        G.add_node(i, text=sent, embedding=emb, source='input', weight=1.0)

    # Compute full similarity matrix in one shot — much faster than nested loops
    sim_matrix = cosine_similarity(embeddings)  # shape: (n, n)

    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            if sim_matrix[i, j] > threshold:
                G.add_edge(i, j, similarity=float(sim_matrix[i, j]))

    return G

## Step 2: Topic Extraction (TF-IDF) + Google Search

In [33]:
import requests
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_topics(text: str, n_topics: int = 5) -> list[str]:
    """
    Use TF-IDF to pull out the most distinctive keywords/phrases.
    These become our search queries.
    Uses 1-2 word n-grams to capture meaningful phrases like
    'climate change' rather than just 'climate'.
    """
    # TF-IDF needs a corpus to compute IDF; with a single doc,
    # it degenerates to TF. Acceptable for keyword extraction.
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        stop_words='english',
        max_features=100
    )
    # Treat sentences as "documents" in our mini-corpus
    sentences = sent_tokenize(text)
    if len(sentences) < 2:
        sentences = [text]
    tfidf_matrix = vectorizer.fit_transform(sentences)
    feature_names = vectorizer.get_feature_names_out()
    # Sum scores across all sentences, pick top n
    scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
    top_indices = scores.argsort()[::-1][:n_topics]
    return [feature_names[i] for i in top_indices]


def google_search(query: str, api_key: str, cse_id: str, num_results: int = 10) -> list[dict]:
    """
    Fetch top num_results results from Google Custom Search API.
    Returns list of result dicts with 'link', 'title', 'snippet'.
    """
    url = "https://www.googleapis.com/customsearch/v1"
    params = {"q": query, "key": api_key, "cx": cse_id, "num": num_results}
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json().get('items', [])


def collect_evidence(text: str, api_key: str, cse_id: str, k: int = 10) -> list[dict]:
    """
    Extract topics from text, search for each, deduplicate results.
    """
    topics = extract_topics(text)
    print(f"Extracted topics: {topics}")
    seen_links = set()
    all_results = []
    for topic in topics:
        results = google_search(topic, api_key, cse_id, num_results=k)
        for r in results:
            if r['link'] not in seen_links:
                seen_links.add(r['link'])
                all_results.append(r)
    return all_results

## Step 3: Cross-Source Agreement Scoring & Document Ranking

The insight: sources that *agree with each other* are more likely to be reporting
factual information than sources that are outliers. We measure this by computing
how similar each document's snippet embedding is to all others (a 'consensus score').

**Limitation to be aware of:** if most sources repeat the same misinformation,
this will rank that misinformation highly. This is an open problem in the field.

In [34]:
def score_documents(documents: list[dict]) -> list[tuple[dict, float]]:
    """
    Rank documents by cross-source agreement.
    Each document's score = average cosine similarity to all other documents.
    High score => this document's content is corroborated by many other sources.
    """
    snippets = [doc.get('snippet', '') for doc in documents]
    if not snippets:
        return []

    embeddings = ENCODER.encode(snippets, batch_size=32, show_progress_bar=False)
    sim_matrix = cosine_similarity(embeddings)  # (n, n)

    scores = []
    n = len(documents)
    for i, doc in enumerate(documents):
        # Average similarity to all *other* documents
        avg_sim = (sim_matrix[i].sum() - 1.0) / max(n - 1, 1)  # exclude self-similarity (=1.0)
        scores.append((doc, float(avg_sim)))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores


# Optional: if you later obtain domain-level reliability scores
# (e.g., from NewsGuard or Media Bias/Fact Check), blend them in here.
def blend_reliability(scored_docs: list[tuple[dict, float]],
                      reliability_scores: dict[str, float],
                      alpha: float = 0.7) -> list[tuple[dict, float]]:
    """
    alpha controls the weight of agreement score vs reliability score.
    alpha=0.7 means 70% agreement, 30% reliability.
    """
    blended = []
    for doc, agreement_score in scored_docs:
        reliability = reliability_scores.get(doc.get('link', ''), 0.5)  # default 0.5
        combined = alpha * agreement_score + (1 - alpha) * reliability
        blended.append((doc, combined))
    blended.sort(key=lambda x: x[1], reverse=True)
    return blended

## Step 4: Knowledge Graph Augmentation with Evidence

In [35]:
def augment_knowledge_graph(G: nx.Graph,
                             scored_docs: list[tuple[dict, float]],
                             sim_threshold: float = 0.75,
                             score_threshold: float = 0.3) -> nx.Graph:
    """
    Add evidence nodes from top-ranked documents into the knowledge graph.
    Edge weights between evidence and existing nodes encode how strongly
    the evidence supports a given claim.

    FIX from v1: We collect all new nodes/edges first, THEN add them.
    Mutating G.nodes while iterating it causes a RuntimeError.
    """
    existing_nodes = list(G.nodes(data=True))  # snapshot before we modify
    next_id = max(G.nodes()) + 1 if G.nodes() else 0

    new_nodes = []  # (id, text, embedding, source_score)
    new_edges = []  # (new_id, existing_id, similarity)

    for doc, doc_score in scored_docs:
        if doc_score < score_threshold:
            continue
        snippet = doc.get('snippet', '')
        if not snippet:
            continue

        evidence_facts = extract_facts(snippet)

        for fact_text, fact_emb in evidence_facts:
            fact_emb_2d = fact_emb.reshape(1, -1)  # sklearn needs (1, d)
            nid = next_id
            next_id += 1
            new_nodes.append((nid, fact_text, fact_emb, doc_score))

            # Connect to similar existing nodes
            for existing_id, data in existing_nodes:
                existing_emb = data['embedding'].reshape(1, -1)
                sim = float(cosine_similarity(fact_emb_2d, existing_emb)[0, 0])
                if sim > sim_threshold:
                    new_edges.append((nid, existing_id, sim, doc_score))

    # Now safely add everything to G
    for nid, text, emb, score in new_nodes:
        G.add_node(nid, text=text, embedding=emb, source='evidence', weight=score)

    for nid, existing_id, sim, doc_score in new_edges:
        G.add_edge(nid, existing_id, similarity=sim, evidence_weight=doc_score)

    return G


def compute_node_confidence(G: nx.Graph) -> dict[int, float]:
    """
    Estimate how well-supported each original claim is.
    For each node, sum the evidence_weight of all edges to 'evidence' nodes.
    Normalize by the number of evidence nodes in the graph.
    """
    n_evidence = sum(1 for _, d in G.nodes(data=True) if d.get('source') == 'evidence')
    confidence = {}
    for node_id, data in G.nodes(data=True):
        if data.get('source') != 'input':
            continue
        total_support = sum(
            G.edges[node_id, nbr].get('evidence_weight', 0)
            for nbr in G.neighbors(node_id)
            if G.nodes[nbr].get('source') == 'evidence'
        )
        confidence[node_id] = total_support / max(n_evidence, 1)
    return confidence

## Step 5: Graph Neural Network Classifier

Instead of averaging embeddings (which throws away graph structure), we use a
Graph Convolutional Network (GCN). The key idea:
- Each node updates its representation by aggregating from its neighbors
- After 2 rounds of this, each node 'knows about' its 2-hop neighborhood
- We then pool all node representations into one graph-level vector
- A linear layer turns that into a fake/real prediction

Requires: `pip install torch-geometric`

In [36]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, DataLoader


def graph_to_pyg(G: nx.Graph, label: int | None = None) -> Data:
    """
    Convert a networkx graph to a PyTorch Geometric Data object.
    Node features = sentence embeddings (optionally scaled by confidence weight).
    """
    node_ids = list(G.nodes())
    id_map = {nid: i for i, nid in enumerate(node_ids)}

    # Node features: embedding weighted by source reliability
    x = torch.tensor(
        np.array([G.nodes[n]['embedding'] * G.nodes[n].get('weight', 1.0)
                  for n in node_ids]),
        dtype=torch.float
    )

    # Edge index in COO format (2 x num_edges) — what PyG expects
    if G.edges():
        edges = [(id_map[u], id_map[v]) for u, v in G.edges()]
        edge_index = torch.tensor(edges, dtype=torch.long).T
        # GCN expects undirected, so add reverse edges
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)

    data = Data(x=x, edge_index=edge_index)
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.float)
    return data


class FakeNewsGCN(torch.nn.Module):
    """
    Two-layer GCN followed by global mean pooling and a binary classifier.

    Architecture:
      GCNConv(384 -> 128) -> ReLU -> Dropout
      GCNConv(128 -> 64)  -> ReLU
      GlobalMeanPool      -> collapses variable-size graph to fixed vector
      Linear(64 -> 1)     -> Sigmoid -> probability of being 'fake'
    """
    def __init__(self, input_dim: int = EMBEDDING_DIM, hidden_dim: int = 128, dropout: float = 0.3):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, 64)
        self.classifier = torch.nn.Linear(64, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)     # (num_graphs_in_batch, 64)
        return torch.sigmoid(self.classifier(x))


def train_gcn(pyg_dataset: list[Data], epochs: int = 20, lr: float = 1e-3) -> FakeNewsGCN:
    model = FakeNewsGCN()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = torch.nn.BCELoss()
    loader = DataLoader(pyg_dataset, batch_size=16, shuffle=True)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in loader:
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch).squeeze()
            loss = criterion(out, batch.y.squeeze())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}  Loss: {total_loss/len(loader):.4f}")
    return model


def evaluate_gcn(model: FakeNewsGCN, pyg_dataset: list[Data]) -> dict:
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    model.eval()
    loader = DataLoader(pyg_dataset, batch_size=16)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(batch.x, batch.edge_index, batch.batch).squeeze()
            all_preds.extend(out.tolist())
            all_labels.extend(batch.y.squeeze().tolist())
    binary_preds = [1 if p > 0.5 else 0 for p in all_preds]
    return {
        'accuracy': accuracy_score(all_labels, binary_preds),
        'f1': f1_score(all_labels, binary_preds),
        'auc': roc_auc_score(all_labels, all_preds)
    }

## Step 6: Model Persistence

Use `torch.save` for PyTorch models, not `joblib`.
`joblib` is designed for sklearn estimators; it doesn't handle PyTorch's computation graph.

In [37]:
def save_model(model: torch.nn.Module, path: str):
    """Save model weights. Saving state_dict (weights only) is preferred
    over saving the whole model object — it's more portable."""
    torch.save(model.state_dict(), path)
    print(f"Saved to {path}")

def load_model(path: str) -> FakeNewsGCN:
    """Reconstruct model and load saved weights."""
    model = FakeNewsGCN()
    model.load_state_dict(torch.load(path, weights_only=True))
    model.eval()
    return model

## Step 7: Dataset Loading & End-to-End Training Loop

In [38]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(os.path.abspath("data/fake_news_dataset.csv"))
df['label_binary'] = (df['label'] == 'fake').astype(int)
print(df[['title', 'label', 'label_binary']].head())
print(f"\nClass balance: {df['label_binary'].value_counts().to_dict()}")

                                  title label  label_binary
0               Foreign Democrat final.  real             0
1   To offer down resource great point.  fake             1
2          Himself church myself carry.  fake             1
3                  You unit its should.  fake             1
4  Billion believe employee summer how.  fake             1

Class balance: {1: 10056, 0: 9944}


In [ ]:
# NOTE: In a full run you'd also augment with Google Search here.
# For offline training, we build graphs from article text alone.

from tqdm import tqdm

def build_dataset_offline(df: pd.DataFrame, text_col='text', label_col='label_binary',
                           sample: int | None = None) -> list[Data]:
    """Build PyG graphs from article text, no external search required."""
    if sample:
        df = df.sample(sample, random_state=42)
    pyg_data = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Building graphs'):
        facts = extract_facts(row[text_col])
        if not facts:
            continue
        G = build_knowledge_graph(facts)
        pyg_data.append(graph_to_pyg(G, label=int(row[label_col])))
    return pyg_data

# Build on a sample first to verify the pipeline works
pyg_dataset = build_dataset_offline(df, sample=200)
print(f"Built {len(pyg_dataset)} graphs")

Building graphs: 100%|██████████| 200/200 [00:09<00:00, 20.90it/s]

Built 200 graphs


In [41]:
# Train/test split at the graph level
train_data, test_data = train_test_split(pyg_dataset, test_size=0.2, random_state=42)

model = train_gcn(train_data, epochs=200)

metrics = evaluate_gcn(model, test_data)
print(f"\nTest Results:")
print(f"  Accuracy: {metrics['accuracy']:.3f}")
print(f"  F1 Score: {metrics['f1']:.3f}")
print(f"  AUC-ROC:  {metrics['auc']:.3f}")

save_model(model, 'fake_news_gcn.pt')

C:\Users\bhada\AppData\Local\Temp\ipykernel_43540\2965857557.py:66: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  loader = DataLoader(pyg_dataset, batch_size=16, shuffle=True)


Epoch 1/200  Loss: 0.6927
Epoch 2/200  Loss: 0.6898
Epoch 3/200  Loss: 0.6856
Epoch 4/200  Loss: 0.6810
Epoch 5/200  Loss: 0.6744
Epoch 6/200  Loss: 0.6679
Epoch 7/200  Loss: 0.6560
Epoch 8/200  Loss: 0.6357
Epoch 9/200  Loss: 0.6148
Epoch 10/200  Loss: 0.5874
Epoch 11/200  Loss: 0.5754
Epoch 12/200  Loss: 0.5160
Epoch 13/200  Loss: 0.4960
Epoch 14/200  Loss: 0.4349
Epoch 15/200  Loss: 0.4225
Epoch 16/200  Loss: 0.3739
Epoch 17/200  Loss: 0.3698
Epoch 18/200  Loss: 0.3541
Epoch 19/200  Loss: 0.3058
Epoch 20/200  Loss: 0.2700
Epoch 21/200  Loss: 0.2457
Epoch 22/200  Loss: 0.2324
Epoch 23/200  Loss: 0.2345
Epoch 24/200  Loss: 0.1875
Epoch 25/200  Loss: 0.1682
Epoch 26/200  Loss: 0.1573
Epoch 27/200  Loss: 0.1268
Epoch 28/200  Loss: 0.1221
Epoch 29/200  Loss: 0.0985
Epoch 30/200  Loss: 0.0981
Epoch 31/200  Loss: 0.0777
Epoch 32/200  Loss: 0.0713
Epoch 33/200  Loss: 0.0573
Epoch 34/200  Loss: 0.0599
Epoch 35/200  Loss: 0.0510
Epoch 36/200  Loss: 0.0476
Epoch 37/200  Loss: 0.0394
Epoch 38/2

C:\Users\bhada\AppData\Local\Temp\ipykernel_43540\2965857557.py:85: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  loader = DataLoader(pyg_dataset, batch_size=16)


## Next Steps & Future Improvements

1. **Relation-typed edges**: Right now all edges mean 'similar'. Train a model to label them
   as 'supports', 'contradicts', or 'neutral' — this dramatically improves the graph's signal.
   Look into Natural Language Inference (NLI) models for this (e.g. `cross-encoder/nli-deberta-v3-small`).

2. **Clustering for consensus**: Instead of pairwise average similarity for document ranking,
   cluster document embeddings (DBSCAN works well here) and rank documents within the largest cluster highest.
   This is more robust to outlier sources.

3. **Reliability scores**: Look into MediaBias/FactCheck API or NewsGuard for domain-level
   credibility scores to blend into your ranking.

4. **Graph Attention Networks (GAT)**: GATConv lets the model *learn* which edges to weight more
   heavily, rather than treating all edges equally. Often outperforms plain GCN.

5. **Class imbalance**: Real datasets often have many more real articles than fake.
   Use `WeightedRandomSampler` in your DataLoader or set `pos_weight` in `BCEWithLogitsLoss`.

6. **Baseline comparison**: Always compare your GCN against a simple TF-IDF + Logistic Regression
   baseline. If your fancy graph approach doesn't beat it, something is wrong (or the dataset is too small).
   This is standard practice in NLP research.